# Predicted Epitopes of Trypanosoma cruzi based on Phage Display Data Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset with the `mlcroissant` library. The dataset contains predicted antigenic epitopes based on phage display data, with extensive metadata and multiple record sets accessible via the Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.etbd-dths/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant matplotlib seaborn

## 1. Data Loading
We start by loading the dataset metadata using `mlcroissant`. This step retrieves key dataset attributes such as name and description.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.etbd-dths/fair2.json"

import sys
import tempfile
import urllib.request

PATCH_URL = 'https://raw.githubusercontent.com/senscience/notebooks/main/tools/patch_fair2.py'
patch_folder = tempfile.mkdtemp()
urllib.request.urlretrieve(PATCH_URL, f'{patch_folder}/patch_fair2.py')
sys.path.insert(0, patch_folder)
from patch_fair2 import load_dataset

# Load the dataset
dataset = load_dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description[:500]}...")

## 2. Data Overview
The dataset comprises several record sets (tables), each with its own schema. We list all available record sets, their `@id`s, and the fields (columns) associated with each. All references are made by their `@id` as required.

In [ ]:
# List available record sets and their @id
record_sets = dataset.metadata.record_sets
print("Record Sets in the Dataset:")
for rs in record_sets:
    print(f"- Name: {rs.name}\n  @id: {rs.id}\n  Description: {rs.description if hasattr(rs, 'description') else 'No description'}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, type: {getattr(field, 'data_type', 'unknown')})")
    print()

## 3. Data Extraction
Load the records of each record set as a pandas DataFrame. In this dataset the fields are identified by UUIDs, and `mlcroissant` names the columns by field `@id` and returns text as bytes: the columns get the field names and the text is decoded.

In [ ]:
def decode(value):
    return value.decode('utf-8') if isinstance(value, bytes) else value

dataframes, loading = {}, []
for rs in record_sets:
    try:
        df = pd.DataFrame(dataset.records(record_set=rs.id))
    except Exception as error:
        loading.append({'Record set': rs.name, 'Rows': None, 'Columns': None,
                        'Status': f'not readable ({type(error).__name__})'})
        continue
    df = df.rename(columns={field.id: field.name for field in rs.fields}).apply(lambda column: column.map(decode))
    dataframes[rs.name] = df
    loading.append({'Record set': rs.name, 'Rows': len(df), 'Columns': df.shape[1], 'Status': 'loaded'})

display(pd.DataFrame(loading).style.hide(axis='index')
        .format({'Rows': '{:,.0f}', 'Columns': '{:.0f}'}, na_rep='—')
        .set_caption('Record sets'))

epitopes = dataframes['epizap_predicted_epitopes.csv']
agrs = dataframes.get('epizap_agrs.csv')

## 4. Exploratory Data Analysis (EDA)
Each predicted epitope (PE) carries the number of phage-display DNA inserts that mapped to it in each of four clinical groups of the study: uninfected **controls**, and Chagas disease patients who are **asymptomatic** or have **mild** or **severe chronic Chagas cardiomyopathy (CCC)**. Each epitope is also matched (BLAST) to its best *T. cruzi* protein, which is classified in a protein group, and to the epitopes already registered in the IEDB.

### 4.1 Overview

In [ ]:
GROUPS = {'control': 'Control', 'asymptomatic': 'Asymptomatic', 'CCC_mild': 'Mild CCC', 'CCC_severe': 'Severe CCC'}
DISEASE = ['asymptomatic', 'CCC_mild', 'CCC_severe']

epitopes['disease_inserts'] = epitopes[DISEASE].sum(axis=1)
epitopes['tcruzi_iedb_match'] = epitopes['tcruzi_iedb_hits'].str.strip() != '[]'
epitopes['human_iedb_match'] = epitopes['human_iedb_hits'].str.strip() != '[]'

overview = {
    'Predicted epitopes': len(epitopes),
    'DNA inserts mapped to the epitopes': epitopes['Number_of_Inserts'].sum(),
    'Antigenic genomic regions (AGRs) of the epitopes, summed': epitopes['Number_of_AGRs'].sum(),
    'Epitopes matching a known T. cruzi epitope (IEDB)': epitopes['tcruzi_iedb_match'].sum(),
    'Epitopes matching a known human epitope (IEDB)': epitopes['human_iedb_match'].sum(),
}
if agrs is not None:
    overview['Distinct AGRs (epizap_agrs.csv)'] = len(agrs)
display(pd.DataFrame({'Value': overview}).style.format('{:,}').set_caption('The dataset at a glance'))

by_group = pd.DataFrame({
    'DNA inserts': epitopes[list(GROUPS)].sum(),
    'Epitopes detected': (epitopes[list(GROUPS)] > 0).sum(),
}).rename(index=GROUPS)
by_group['Share of all inserts (%)'] = 100 * by_group['DNA inserts'] / by_group['DNA inserts'].sum()
display(by_group.style.format({'DNA inserts': '{:,}', 'Epitopes detected': '{:,}', 'Share of all inserts (%)': '{:.1f}'})
        .set_caption('DNA inserts by clinical group'))

The clinical groups were not sequenced to the same depth: the control group holds about 7% of the inserts. Raw counts therefore overstate every epitope in the disease groups; the comparisons below use each group's **share of its own inserts** instead.

### 4.2 Grouping: protein groups
The epitopes by protein group of their best-matching *T. cruzi* protein. The last four columns are normalized: the percentage of each clinical group's inserts that falls in the protein group, so that groups of different sequencing depth compare.

In [ ]:
families = epitopes.groupby('Protein_group').agg(**{
    'Epitopes': ('ID', 'size'),
    'Median identity (%)': ('identity', 'median'),
    'T. cruzi IEDB matches': ('tcruzi_iedb_match', 'sum'),
})
families.insert(1, 'Share of epitopes (%)', 100 * families['Epitopes'] / len(epitopes))
share = 100 * epitopes.groupby('Protein_group')[list(GROUPS)].sum() / epitopes[list(GROUPS)].sum()
families = families.join(share.rename(columns=lambda column: f'{GROUPS[column]} (% of inserts)'))
families = families.sort_values('Epitopes', ascending=False)
families.index.name = 'Protein group'

share_columns = [column for column in families.columns if column.endswith('(% of inserts)')]
display(families.style
        .format('{:.1f}', subset=['Share of epitopes (%)', 'Median identity (%)'] + share_columns, na_rep='—')
        .format('{:,}', subset=['Epitopes', 'T. cruzi IEDB matches'])
        .background_gradient(cmap='Blues', subset=share_columns, vmin=0, vmax=100)
        .set_caption('Epitopes and inserts by protein group'))

Hypothetical proteins hold most epitopes and most of the control inserts. The "Others" group is the striking difference between the clinical groups: it holds a small share of the control inserts but the largest share of the asymptomatic and severe CCC inserts. `NO HIT` epitopes have no BLAST match, hence no identity.

### 4.3 Filtering: epitopes never seen in the control group
Epitopes with inserts in the disease groups but none in the control group are candidate disease-specific antigens. The ten with the most inserts in the disease groups:

In [ ]:
candidates = epitopes[(epitopes['control'] == 0) & (epitopes['disease_inserts'] > 0)]
print(f'{len(candidates):,} of {len(epitopes):,} epitopes have inserts in the disease groups and none in the control group.')

top = (candidates.nlargest(10, 'disease_inserts')
       [['PE', 'Protein_group', 'protein_description', *DISEASE, 'identity', 'tcruzi_iedb_match']]
       .rename(columns={'PE': 'Epitope sequence', 'Protein_group': 'Protein group',
                        'protein_description': 'Best-matching protein', 'identity': 'Identity (%)',
                        'tcruzi_iedb_match': 'Known T. cruzi epitope', **GROUPS}))
top['Known T. cruzi epitope'] = top['Known T. cruzi epitope'].map({True: 'yes', False: ''})
display(top.style.hide(axis='index')
        .format({'Identity (%)': '{:.1f}', **{GROUPS[group]: '{:,}' for group in DISEASE}}, na_rep='—')
        .set_caption('Top 10 epitopes never seen in the control group'))

## 5. Visualization
- **Clinical group comparison**: each epitope by its inserts in the disease groups (x) and in the control group (y), both per million inserts of their side, so that the diagonal marks an epitope as abundant in patients as in controls. Epitopes below the diagonal are enriched in patients; those never seen in controls lie on the bottom edge. The marker size is the number of peptides of the epitope, and the colour its protein group (the six annotated protein families; hypothetical, other and unmatched proteins in grey).
- **Protein group distribution**: the number of epitopes per protein group.
- **Sequence identity distribution**: the BLAST identity of each epitope to its best-matching *T. cruzi* protein (`NO HIT` epitopes have none).

In [ ]:
FAMILIES = ['TS', 'DGF', 'RHS', 'MASP', 'MUC', 'GP63']
COLORS = dict(zip(FAMILIES, ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#6250d6']))
CONTEXT = '#c3c2b7'   # hypothetical, other and unmatched proteins

fig = plt.figure(figsize=(16, 9))
grid = fig.add_gridspec(2, 2, width_ratios=[1.5, 1], hspace=0.35, wspace=0.25)

# Clinical group comparison, in inserts per million of each side (disease groups / control group)
ax = fig.add_subplot(grid[:, 0])
x = np.log10(1e6 * epitopes['disease_inserts'] / epitopes['disease_inserts'].sum() + 1)
y = np.log10(1e6 * epitopes['control'] / epitopes['control'].sum() + 1)
size = 6 + 4 * np.sqrt(epitopes['Number_of_Peptides'])
context = ~epitopes['Protein_group'].isin(FAMILIES)
ax.scatter(x[context], y[context], s=size[context], color=CONTEXT, alpha=0.5,
           edgecolor='white', linewidth=0.4, label=f'Hypothetical, others, no hit ({context.sum():,})')
for family in FAMILIES:
    selected = epitopes['Protein_group'] == family
    ax.scatter(x[selected], y[selected], s=size[selected], color=COLORS[family], alpha=0.85,
               edgecolor='white', linewidth=0.4, label=f'{family} ({selected.sum():,})')
limit = max(x.max(), y.max()) * 1.03
ax.plot([0, limit], [0, limit], linestyle='--', linewidth=1, color='#898781', label='As abundant in both')
ax.set(xlim=(0, limit), ylim=(0, limit), title='Clinical group comparison',
       xlabel='Disease groups: log10(inserts per million + 1)',
       ylabel='Control group: log10(inserts per million + 1)')
legend = ax.legend(title='Protein group (marker size: number of peptides)', loc='upper left', fontsize=9)
for handle in legend.legend_handles:
    if hasattr(handle, 'set_sizes'):
        handle.set_sizes([40])

# Protein group distribution
ax = fig.add_subplot(grid[0, 1])
counts = epitopes['Protein_group'].value_counts()
bars = ax.bar(counts.index, counts.values, color=[COLORS.get(group, CONTEXT) for group in counts.index],
              edgecolor='white', linewidth=2)
ax.bar_label(bars, labels=[f'{value:,}' for value in counts.values], fontsize=8, padding=2)
ax.set(title='Protein group distribution', ylabel='Epitopes')
ax.tick_params(axis='x', rotation=45)
ax.margins(y=0.12)

# Sequence identity distribution
ax = fig.add_subplot(grid[1, 1])
ax.hist(epitopes['identity'].dropna(), bins=np.arange(60, 102.5, 2.5), color='#2a78d6',
        edgecolor='white', linewidth=1)
ax.set(title='Sequence identity distribution', xlabel='Identity to the best-matching T. cruzi protein (%)',
       ylabel='Epitopes')

plt.show()

both = (epitopes['control'] > 0) & (epitopes['disease_inserts'] > 0)
print(f"{both.sum():,} epitopes are seen in both the disease and the control groups; only {(x[both] > y[both]).mean():.0%} "
      f"of them are relatively more abundant in the disease groups.")
print(f"{len(candidates):,} epitopes are seen only in the disease groups (bottom edge), "
      f"{(epitopes['disease_inserts'] == 0).sum():,} only in the control group (left edge).")
print(f"{(epitopes['identity'] == 100).sum():,} epitopes match their best T. cruzi protein with 100% identity; "
      f"{epitopes['identity'].isna().sum():,} have no match (NO HIT).")

## 6. Conclusion
We loaded the FAIR² *T. cruzi* predicted epitope dataset with the `mlcroissant` library and explored its two readable record sets:

- **Overview**: 7,811 predicted epitopes, mapped by more than 2.7 million phage-display DNA inserts from four clinical groups sequenced to very different depths, hence the normalized comparisons.
- **Protein groups**: most epitopes come from hypothetical proteins, and the patients' inserts are concentrated in different protein groups from the controls' ones.
- **Disease-specific candidates**: over half of the epitopes are never seen in the control group; the ten most abundant of them are listed, with their sequence and best-matching protein.
- **Visualization**: once normalized, the epitopes fall into two populations: those seen in both patients and controls, which are mostly relatively more abundant in the controls, and those seen only in patients, the candidate disease-specific antigens. Most epitopes match their best *T. cruzi* protein with high identity, a third of those with a match at 100%.

The third record set, the genome assembly statistics (`trypanosoma_cruzi_brazil_a4_ASM1503362v1_assembly_stats.tsv`), cannot be read by `mlcroissant` from the published metadata. Further exploration can compare the three disease groups with each other (asymptomatic vs mild vs severe CCC), or map the epitopes on the genome with their antigenic genomic regions (`epizap_agrs.csv`).